# MOE from Scratch

In this tutorial, we will first implement a MOE layer in PyTorch. Then we expand it to a distributed version using PyTorch's `torch.distributed` package. Finally, we will implement a MOE kernel in Triton.


In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA is not available"
torch.set_default_device('cuda')



## MOE in PyTorch

Reference:
- [Inside OpenAI’s gpt-oss: How Production MoE Really Works](https://medium.com/@chris.p.hughes10/inside-openais-gpt-oss-how-production-moe-really-works-cfa5f6a23caa)
- [openai/gpt-oss-120b](https://huggingface.co/openai/gpt-oss-120b)

### Tensor Data Flow (Token centric)

Activation:
  [S, top_k, 2*d_intermediate] → [S, top_k, d_intermediate]

Step 7 - Second Expert Layer (einsum + bias):
  [S, top_k, d_hidden, d_intermediate] @ [S, top_k, d_intermediate] → [S, top_k, d_hidden]

Step 8 - Distributed All-Reduce (if world_size > 1):
  [S, top_k, d_hidden] → [S, top_k, d_hidden]

Step 9 - Weighted Expert Combination (einsum):
  [S, top_k, d_hidden] weighted by [S, top_k] → [S, d_hidden]
```txt
Input:
  x: [S, d_hidden] (S = b*seq_len, flattened batch size)

Step 1 - Normalization:
  [S, d_hidden] → [S, d_hidden]

Step 2 - Expert Scoring (gate network):
  [S, d_head] → [S, num_experts]

Step 3 - Top-K Selection:
  [S, num_experts] → [S, top_k] expert indices + [S, tok_k] weights

Step 4 - Weight Extraction (advanced indexing):
  mlp1_weight: [num_experts, 2*d_intermediate, d_hidden] → [S, top_k, 2*d_intermediate, d_hidden]
  mlp2_weight: [num_experts, d_hidden, d_intermediate] → [S, top_k, d_hidden, d_intermediate]

Step 5 - First Expert Layer (einsum + bias):
  [S, top_k, 2*d_intermediate, d_hidden] @ [S, d_hidden] → [S, top_k, 2*d_intermediate]

Step 6 - SwiGLU 
Step 10 - Residual Connection:
  [S, d_hidden] + [S, d_hidden] → [S, d_hidden]

Output:
  [S, d_hidden]

```


In [ ]:
from dataclasses import dataclass

@dataclass
class ModelConfig:
    num_hidden_layers: int = 36
    num_experts: int = 128
    experts_per_token: int = 4
    vocab_size: int = 201088
    hidden_size: int = 2880
    intermediate_size: int = 2880
    swiglu_limit: float = 7.0
    head_dim: int = 64
    num_attention_heads: int = 64
    num_key_value_heads: int = 8
    sliding_window: int = 128
    initial_context_length: int = 4096
    rope_theta: float = 150000.0
    rope_scaling_factor: float = 32.0
    rope_ntk_alpha: float = 1.0
    rope_ntk_beta: float = 32.0


# Utility for RMSNorm
class RMSNorm(torch.nn.Module):
    def __init__(
        self, num_features: int, eps: float = 1e-05, device: torch.device | None = None
    ):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.scale = torch.nn.Parameter(
            torch.ones(num_features, device=device, dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-1] == self.num_features
        t, dtype = x.float(), x.dtype
        t = t * torch.rsqrt(torch.mean(t**2, dim=-1, keepdim=True) + self.eps)
        return (t * self.scale).to(dtype)

In [ ]:
class MLPBlock(torch.nn.Module):
    def __init__(self, config: ModelConfig, device: torch.device):
        self.norm = RMSNorm(config.hidden_size, device=device)
        self.gate = torch.nn.Linear(config.hidden_size, config.num_experts, device=device, dtype=torch.bfloat16)
        self.world_size = 1
        self.config = config
        
        self.mlp1_weight = torch.nn.Parameter(
            torch.empty((config.num_experts, 2 * config.intermediate_size // self.world_size, config.hidden_size),
                        device=device, 
                        dtype=torch.bfloat16)
        )
        self.mlp1_bias = torch.nn.Parameter(
            torch.empty((config.num_experts, 2 * config.intermediate_size // self.world_size),
                        device=device,
                        dtype=torch.bfloat16)
        )

        self.mlp2_weight = torch.nn.Parameter(
            torch.empty((config.num_experts, config.hidden_size, config.intermediate_size),
                        device=device,
                        dtype=torch.bfloat16)
        )
        self.mlp2_bias = torch.nn.Parameter(
            torch.empty((config.num_experts, config.hidden_size),
                        device=device,
                        dtype=torch.bfloat16)
        )
    
    def _swiglu(self, x: torch.Tensor, alpha: float = 1.702):
        # x in shape [b, seq, 2*intermediate]
        x_glu = x[..., 0::2]
        x_linear = x[..., 1::2]
        
        x_glu = torch.clamp(x_glu, max=self.config.swiglu_limit)
        x_linear = torch.clamp(x_linear, min=-self.config.swiglu_limit,  max=self.config.swiglu_limit)
        
        # Custom formula: sigmoid(α·gate) * gate * (linear + 1)
        # Compare to standard: gate * silu(up)
        out_glu = x_glu * torch.sigmoid(alpha * x_glu)
        return out_glu * (x_linear + 1)
        
    
    def forward_demo(self, x: torch.Tensor):
        residule_x = x.clone()
        x = self.norm(x) # [s, d_hidden]
        expert_score = self.gate(x) # [s, experts]
        top_k_expert_score, tok_k_expert_idx = torch.topk(expert_score, self.config.experts_per_token, dim=-1) # [s, top_k]
        top_k_expert_score = torch.nn.functional.softmax(top_k_expert_score, dim=-1)
        
        mlp1_weight = self.mlp1_weight[tok_k_expert_idx, ...] # [s, tok_k, 2*d_intermediate, d_hidden]
        mlp1_bias = self.mlp1_bias[tok_k_expert_idx, ...] # [s, tok_k, intermediate]

        x = torch.einsum("bcdk, bk->bcd", mlp1_weight, x)
        x += mlp1_bias
        x = self._swiglu(x) # [s, tok_k, intermediate]
        
        mlp2_weight = self.mlp2_weight[tok_k_expert_idx, ...] # [s, tok_k, hidden, intermediate]
        mlp2_bias = self.mlp2_bias[tok_k_expert_idx, ...]

        x = torch.einsum("bcdk, bck->bcd", mlp2_weight, x)
        x += mlp2_bias # [s, tok_k, hidden]
        # score [s, tok_k]
        x = torch.einsum("bk, bkc->bc", top_k_expert_score, x)

        return residule_x + x    
    
                   
        


### Tensor Data Flow (Expert centric)

```txt
Input:
  x: [S, d_hidden] (S = b*seq_len, flattened batch size)

Step 1 - Normalization:
  [S, d_hidden] → [S, d_hidden]

Step 2 - Expert Scoring (gate network):
  [S, d_hidden] → [S, num_experts]

Step 3 - Top-K Selection:
  [S, num_experts] → [S, top_k] expert indices + [S, top_k] weights

Step 4 - Token Grouping & Masking (The Loop Initiation):
  For each expert_id in unique_experts:
    mask = (indices == expert_id) -> [S, top_k] (Boolean)
    selected_idx = mask.any(dim=-1).nonzero() -> [N_expert] (N_expert 为当前专家分到的 Token 数)

Step 5 - Input Extraction (index_select):
  expert_input: x_flat[selected_idx] -> [N_expert, d_hidden]

Step 6 - Expert FFN Layer 1 (Up-projection):
  [N_expert, d_hidden] @ [d_hidden, 2*d_intermediate].T → [N_expert, 2*d_intermediate]
  (注：此处权重为静态 [2*d_intermediate, d_hidden]，无需像 Token-Centric 那样进行高级索引提取)

Step 7 - Activation (SwiGLU/Gating):
  [N_expert, 2*d_intermediate] → [N_expert, d_intermediate]

Step 8 - Expert FFN Layer 2 (Down-projection):
  [N_expert, d_intermediate] @ [d_intermediate, d_hidden].T → [N_expert, d_hidden]

Step 9 - Weighted Scatter & Accumulation (index_add):
  out_flat[selected_idx] += (expert_out * weights[mask]) -> [S, d_hidden]
  (注：在循环中不断根据对应的 Top-k 权重累加回结果表)

Step 10 - Residual Connection:
  [S, d_hidden] + [S, d_hidden] → [S, d_hidden]

Output:
  [S, d_hidden]
```

In [ ]:
# Source: LLMs-from-scratch (Sebastian Raschka)
# ch05/11_qwen3/standalone-qwen3-moe-plus-kvcache.ipynb

class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Each expert is a separate nn.Linear module
        self.fc1 = nn.ModuleList([nn.Linear(...) for _ in range(num_experts)])
        self.fc2 = nn.ModuleList([nn.Linear(...) for _ in range(num_experts)])
        self.fc3 = nn.ModuleList([nn.Linear(...) for _ in range(num_experts)])
    
    def forward(self, x):
        scores = self.gate(x)
        topk_scores, topk_indices = torch.topk(scores, num_experts_per_tok, dim=-1)
        topk_probs = torch.softmax(topk_scores, dim=-1)
        
        # Flatten batch and sequence dimensions
        x_flat = x.reshape(batch * seq_len, -1)
        out_flat = torch.zeros_like(x_flat)
        
        # Loop over each unique expert that was selected
        unique_experts = torch.unique(topk_indices_flat)
        for expert_id in unique_experts:
            # Find which tokens need this expert
            mask = topk_indices_flat == expert_id
            selected_idx = mask.any(dim=-1).nonzero().squeeze(-1)
            
            # Process all tokens that selected this expert
            expert_input = x_flat.index_select(0, selected_idx)
            hidden = silu(self.fc1[expert_id](expert_input)) * self.fc2[expert_id](expert_input)
            expert_out = self.fc3[expert_id](hidden)
            
            # Accumulate weighted results
            out_flat.index_add_(0, selected_idx, expert_out * weights)
        
        return out_flat.reshape(batch, seq_len, emb_dim)